In [1]:
import asyncio
import importlib
import os
import sys
import threading


sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), "../")))

from pprint import pprint

import things

import hololinked


importlib.reload(hololinked)
importlib.reload(things)

<module 'things' from 'd:\\current projects\\hololinked-main\\tests\\things\\__init__.py'>

In [2]:
from things import OceanOpticsSpectrometer, TestThing


thing = TestThing(id="example-test")
spectrometer = OceanOpticsSpectrometer(id="example-spectrometer")

21-02-2026T07:00:40.927827Z [info     ] [THING] initialialised Thing class TestThing with id example-test Thing=TestThing thing_id=example-test
21-02-2026T07:00:40.929474Z [info     ] [THING] setup state machine, states=['DISCONNECTED', 'ON', 'FAULT', 'MEASURING', 'ALARM'], initial_state=DISCONNECTED Thing=OceanOpticsSpectrometer thing_id=example-spectrometer
21-02-2026T07:00:40.930055Z [info     ] [THING] initialialised Thing class OceanOpticsSpectrometer with id example-spectrometer Thing=OceanOpticsSpectrometer thing_id=example-spectrometer


In [12]:
pprint(thing.get_thing_model(ignore_errors=True).json(), indent=4)

20-02-2026T18:38:22.861693Z [error    ] [THING] Error while generating schema for base_property - WoT schema generator for this descriptor/property is not implemented. name base_property & type <class 'hololinked.core.property.Property'> Thing=TestThing thing_id=example-test
{   'actions': {   'action_echo': {'synchronous': True},
                   'execute_instruction': {   'description': 'executes '
                                                             'instruction '
                                                             'given by the '
                                                             'ASCII string '
                                                             'parameter '
                                                             "'command'.If "
                                                             'return data size '
                                                             'is greater than '
                                                    

In [ ]:
# not always True
assert thing.rpc_server is not None

pprint(
    thing.rpc_server.get_thing_description(
        thing, protocol="INPROC", ignore_errors=True,
    ),
    indent=4,
)

In [ ]:
TestThing.get_analogue_offset.to_affordance().json()

In [ ]:
TestThing.set_channel.to_affordance().json()

In [ ]:
TestThing.set_sensor_model.to_affordance().json()

In [ ]:
TestThing.set_channel_pydantic.to_affordance().json()

In [ ]:
from hololinked.utils import get_input_model_from_signature


get_input_model_from_signature(TestThing.set_channel_pydantic.obj)

In [ ]:
TestThing.start_acquisition.to_affordance().json()

In [ ]:
TestThing.execute_instruction.to_affordance().json()

In [ ]:
TestThing.data_point_event.to_affordance().json()

In [ ]:
TestThing.numpy_action.to_affordance().json()

In [ ]:
import json

import httpx

from hololinked.client.security import OAuthDirectAccessGrant


req_rep_timeout = httpx.Timeout(
    connect=10,
    read=10,
    write=10,
    pool=2,
)

req_rep_sync_client = httpx.Client(timeout=req_rep_timeout)
req_rep_async_client = httpx.AsyncClient(timeout=req_rep_timeout)

config = json.loads(open("oidc-config.json").read())

direct_access_grant = OAuthDirectAccessGrant(
    oidc_config_url=config["issuer"] + "/.well-known/openid-configuration",
    client_id=config["client_id"],
    client_secret=config["client_secret"],
    username=config["username"],
    password=config["password"],
)

In [7]:
from hololinked.client import ClientFactory


object_proxy_http = ClientFactory.http(
    "http://localhost:8081/test-spectrometer/resources/wot-td", 
    ignore_TD_errors=True,
    # security=direct_access_grant
)
object_proxy_http.td

{'context': ['https://www.w3.org/2022/wot/td/v1.1'],
 'id': 'test-spectrometer',
 'title': 'OceanOpticsSpectrometer',
 'description': 'OceanOptics spectrometers Test Thing.',
 'properties': {'background_correction': {'description': 'set True for Seabreeze internal black level correction',
   'oneOf': [{'oneOf': [{'type': 'string'}], 'enum': ['AUTO', 'CUSTOM', None]},
    {'type': 'null'}],
   'forms': [{'href': 'http://127.0.0.1:8081/test-spectrometer/background-correction',
     'op': 'readproperty',
     'htv:methodName': 'GET',
     'contentType': 'application/json'},
    {'href': 'http://127.0.0.1:8081/test-spectrometer/background-correction',
     'op': 'writeproperty',
     'htv:methodName': 'PUT',
     'contentType': 'application/json'}]},
  'custom_background_intensity': {'oneOf': [{'type': 'array',
     'items': {'oneOf': [{'type': 'number'}, {'type': 'integer'}]}},
    {'type': 'null'}],
   'forms': [{'href': 'http://127.0.0.1:8081/test-spectrometer/custom-background-intensit

In [15]:
object_proxy_http._security.logout()

In [8]:
import ssl

from hololinked.client import ClientFactory


mqtt_ssl = ssl.create_default_context(ssl.Purpose.SERVER_AUTH)
if not os.path.exists("ca.crt"):
    raise FileNotFoundError(
        "CA certificate 'ca.crt' not found in current directory for MQTT TLS connection"
    )
mqtt_ssl.load_verify_locations(cafile="ca.crt")
mqtt_ssl.check_hostname = True
mqtt_ssl.verify_mode = ssl.CERT_REQUIRED
mqtt_ssl.minimum_version = ssl.TLSVersion.TLSv1_2

object_proxy_mqtt = ClientFactory.mqtt(hostname="localhost", port=8883, thing_id="test-spectrometer", ssl_context=mqtt_ssl, username="sampleuser", password="samplepass")
object_proxy_mqtt.td


{'context': ['https://www.w3.org/2022/wot/td/v1.1'],
 'id': 'test-spectrometer',
 'title': 'OceanOpticsSpectrometer',
 'description': 'OceanOptics spectrometers Test Thing.',
 'properties': {'integration_time': {'description': 'integration time of measurement in milliseconds',
   'default': 1000,
   'type': 'number',
   'observable': True,
   'minimum': 0.001,
   'forms': [{'href': 'mqtts://localhost:8883',
     'op': 'observeproperty',
     'mqv:topic': 'test-spectrometer/integration_time',
     'contentType': 'application/json'}]},
  'state': {'description': "Current state machine's state if state machine present, `None` indicates absence of state machine.State machine returned state is always a string even if specified as an Enum in the state machine.",
   'readOnly': True,
   'oneOf': [{'type': 'string'}, {'type': 'null'}],
   'observable': True,
   'forms': [{'href': 'mqtts://localhost:8883',
     'op': 'observeproperty',
     'mqv:topic': 'test-spectrometer/state',
     'contentT

In [ ]:
from hololinked.client.factory import ClientFactory


object_proxy_zmq = ClientFactory.zmq(
    server_id="example-test",
    thing_id="example-test",
    protocol="IPC",
    ignore_TD_errors=True,
)
pprint(object_proxy_zmq.td, indent=4)

In [9]:
def cb(value):
    print("event0", threading.get_ident(), value.data)


def cb1(value):
    print("event1 ", threading.get_ident(), value.data)


def cb2(value):
    print("event2 ", threading.get_ident(), value.data)


async def async_cb1(value):
    print("event1 ", asyncio.current_task().get_name(), value.data)


async def async_cb2(value):
    print("event2 ", asyncio.current_task().get_name(), value.data)


# object_proxy.subscribe_event("test_event", cb)
# object_proxy.subscribe_event("test_event", [cb1, cb2], concurrent=True)
object_proxy_mqtt.subscribe_event("intensity_measurement_event", [async_cb1, async_cb2], asynch=True)
# object_proxy.subscribe_event("test_event", [async_cb1, async_cb2], asynch=True, concurrent=True)

event1  Task-94 {'value': [11068, 11485, 4490, 11956, 7363, 7588, 15700, 16188, 9257, 4615, 1332, 9384, 1684, 13295, 6046, 4065, 7070, 3292, 9545, 6170, 12937, 643, 9860, 3804, 8619, 3483, 8659, 4285, 6044, 4680, 1086, 643, 9583, 14307, 975, 3963, 13230, 278, 9033, 12653, 15575, 2192, 15585, 3032, 11378, 9456, 2811, 7666, 10020, 14422], 'timestamp': '27.02.2026 21:29:56.718'}
event2  Task-94 {'value': [11068, 11485, 4490, 11956, 7363, 7588, 15700, 16188, 9257, 4615, 1332, 9384, 1684, 13295, 6046, 4065, 7070, 3292, 9545, 6170, 12937, 643, 9860, 3804, 8619, 3483, 8659, 4285, 6044, 4680, 1086, 643, 9583, 14307, 975, 3963, 13230, 278, 9033, 12653, 15575, 2192, 15585, 3032, 11378, 9456, 2811, 7666, 10020, 14422], 'timestamp': '27.02.2026 21:29:56.718'}
event1  Task-94 {'value': [6523, 804, 5292, 5079, 15655, 4245, 15954, 7529, 8824, 7294, 6719, 8992, 14987, 3932, 8099, 16245, 8618, 9183, 11023, 7063, 5935, 8849, 5232, 14601, 2941, 2502, 10326, 5289, 9558, 12455, 14997, 12866, 3948, 14409, 1

In [13]:
object_proxy_http.start_acquisition_single()

In [ ]:
object_proxy_mqtt.subscribe_event("test_event", cb)

In [ ]:
object_proxy_mqtt.subscribe_event(
    "test_event", [async_cb1, async_cb2], asynch=True, concurrent=True
)

In [ ]:
object_proxy_mqtt.subscribe_event(
    "test_binary_payload_event", callbacks=[cb1, cb2], concurrent=True
)

In [ ]:
object_proxy_http.push_events()

In [ ]:
object_proxy_http.push_events(event_name="test_binary_payload_event")

In [ ]:
object_proxy.unsubscribe_event("test_event")

In [ ]:
for i in range(5):
    print(object_proxy.read_property("db_init_int_prop"), i)

In [ ]:
object_proxy_mqtt.read_property("db_init_int_prop")

In [ ]:
print(object_proxy_http.get_non_remote_number_prop())
object_proxy_http.set_non_remote_number_prop(25)
object_proxy_http.get_non_remote_number_prop()

In [23]:
object_proxy_http.get_transports()

AttributeError: 'RPCServer' object has no attribute 'ipc_server'

In [ ]:
# object_proxy.get_serialized_data()
object_proxy_zmq.get_serialized_data()

In [ ]:
val, binary_val = object_proxy_zmq.get_mixed_content_data()
print(val)
print(binary_val)

In [ ]:
# print(object_proxy.numpy_array_prop)
# print(type(object_proxy.numpy_array_prop))
print(object_proxy.numpy_action(object_proxy.numpy_array_prop))

In [ ]:
from hololinked.server.security import APIKeySecurity


auth = APIKeySecurity(name="test5", file="apikeys.json")
api_key = auth.create()

In [ ]:
from hololinked.server.security import APIKeySecurity


loaded_auth = APIKeySecurity(name="test5", file="apikeys.json")

In [ ]:
from datetime import datetime


loaded_auth.prehashed_apikey_expiry = datetime.fromisoformat(
    "2025-01-20T09:39:05.804308"
)

In [ ]:
loaded_auth.validate_input("...")

In [ ]:
from hololinked.server.security import KeycloakOIDCSecurity


auth = KeycloakOIDCSecurity(
    oidc_server_url="http://localhost:8080",
    oidc_realm="hololinked-test",
    oidc_client_id="device-server",
    verify_ssl=False,
    allowed_roles=["device-admin"],
)

token = ""
auth.validate_input(token)
userinfo = auth.userinfo(token)
print("userinfo", userinfo)

auth.user_has_role(userinfo)